# Moontower API Explorer

Scratchpad for hitting every endpoint on `https://api.moontower.ai`.

**Setup** (one-time):
1. `cp .env.example .env` and paste your API key
2. `pip install -r requirements.txt`
3. Run the setup cell below

Each endpoint has its own section — all self-contained, jump to whatever you want.

**API reference:**
- Base: `https://api.moontower.ai/v1`
- Auth: `X-API-Key` header
- Limits: 100 tickers, 31-day range, 30s timeout, 1000 req/min
- Dates: `YYYY-MM-DD`; priority = `trade_date` > `start_date`/`end_date` > latest
- Multi-ticker: repeat the param (`?ticker=SPY&ticker=QQQ`)

## Setup

In [13]:
from datetime import date, timedelta
import pandas as pd
from moontower import call, to_df

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

TICKERS = ["SPY"]
TODAY = date.today().isoformat()
YESTERDAY = (pd.Timestamp.today().normalize() - pd.offsets.BDay(1)).date().isoformat()
_month_ago_ts = pd.Timestamp.today().normalize() - pd.DateOffset(months=1)
if _month_ago_ts.weekday() >= 5:
    _month_ago_ts = _month_ago_ts - pd.offsets.BDay(1)
MONTH_AGO = _month_ago_ts.date().isoformat()

print(f"Ready. TICKERS={TICKERS}  MONTH_AGO={MONTH_AGO}  YESTERDAY={YESTERDAY}  TODAY={TODAY}")

Ready. TICKERS=['SPY']  MONTH_AGO=2026-03-27  YESTERDAY=2026-04-24  TODAY=2026-04-27


## Reference Data

### `/v1/tickers`

Directory of every ticker the API covers. Run this first to see what's available — the result includes symbol, name, asset class, and any grouping tags. Good sanity check that your key works.

In [3]:
tickers_df = to_df(call("tickers"))
print(f"{len(tickers_df)} tickers")
tickers_df.head(20)

431 tickers


,ticker,ticker_id,category,category_id,min_date,max_date
0,AA,403,Metals,9,2007-01-03,None
1,AAL,236,Airlines,29,2013-12-10,None
2,AAPL,92,Technology,13,2007-01-03,None
3,ABBV,223,Pharma,47,2013-01-02,None
4,ABNB,136,Travel,39,None,None
5,ABT,237,Healthcare,23,2007-01-03,None
6,ACHR,423,Transportation,46,2021-09-17,None
7,ADBE,238,Software,32,2007-01-03,None
8,ADI,226,Semiconductors,31,2007-01-03,None
9,AEL,445,Financials,7,2007-01-03,None


## Market Data

### `/v1/price`

OHLCV plus mid. Intraday is 15-min delayed. Accepts a single ticker, multiple tickers (repeat the param), a date range via `start_date`/`end_date`, or a single `trade_date`.

In [4]:
to_df(call("price", ticker=["SPY", "QQQ", "IWM"]))

,ticker,date,snapshot_at,updated_at,open,high,low,close,volume,mid
0,IWM,2026-04-27,2026-04-27T20:00:00,2026-04-27T20:55:02.454000,276.820007,278.239990,276.250000,277.140015,22180364.0,277.244995
1,QQQ,2026-04-27,2026-04-27T20:00:00,2026-04-27T20:55:25.817000,663.400024,664.419983,660.690002,664.229980,31899468.0,662.554993
2,SPY,2026-04-27,2026-04-27T20:00:00,2026-04-27T20:55:26.478000,713.169983,715.609985,712.294983,715.169983,32359176.0,713.952484


In [5]:
to_df(call("price", ticker="SPY", start_date=MONTH_AGO, end_date=TODAY)).tail()

""


In [6]:
to_df(call("price", ticker="SPY", trade_date="2025-12-15"))

,ticker,date,snapshot_at,updated_at,open,high,low,close,volume,mid
0,SPY,2025-12-15,2025-12-15T21:00:00,2025-12-16T05:03:38.516000,685.74,685.76,679.25,680.73,90810898.0,682.505


### `/v1/impliedvol`

Full implied-vol surface by delta level for every listed expiry. Narrow to a specific expiry with `expiry_date` — otherwise you'll get every expiry on that trade date.

In [15]:
iv = to_df(call("impliedvol", ticker="SPY", trade_date=MONTH_AGO))
iv.head()

,ticker,date,expiry_date,dte,spot_price,vol100,vol90,vol80,vol70,vol60,vol50,vol40,vol30,vol20,vol10,vol0,snapshot_at,updated_at
0,SPY,2026-03-27,2026-03-27,0,634.68,26.62,24.10,24.19,24.76,25.63,26.74,28.08,29.66,31.56,34.00,39.32,2026-03-27T20:00:05,2026-03-28T01:05:31.364000
1,SPY,2026-03-27,2026-03-30,3,634.68,31.28,25.41,23.82,23.00,22.52,22.23,22.04,21.90,21.81,21.80,22.82,2026-03-27T20:00:05,2026-03-28T01:05:31.364000
2,SPY,2026-03-27,2026-03-31,4,634.68,37.54,29.21,27.16,26.22,25.76,25.51,25.35,25.16,24.89,24.49,24.84,2026-03-27T20:00:05,2026-03-28T01:05:31.364000
3,SPY,2026-03-27,2026-04-01,5,634.68,32.66,29.33,28.26,27.62,27.16,26.78,26.39,25.94,25.35,24.49,22.66,2026-03-27T20:00:05,2026-03-28T01:05:31.364000
4,SPY,2026-03-27,2026-04-02,6,634.68,35.04,30.83,29.50,28.73,28.18,27.69,27.18,26.56,25.71,24.48,22.01,2026-03-27T20:00:05,2026-03-28T01:05:31.364000


In [8]:
to_df(call("impliedvol", ticker="SPY", trade_date=MONTH_AGO, expiry_date="2026-06-19"))

""


### `/v1/realvol`

Realized volatility at multiple windows (10d/20d/30d/etc).

**⚠️ EOD-only.** Intraday calls return empty — use `trade_date=<yesterday>` or an explicit end-of-day date.

In [14]:
to_df(call("realvol", ticker="SPY", trade_date=YESTERDAY))

,ticker,date,snapshot_at,updated_at,rv_1d,rv_7d,rv_14d,rv_30d,rv_60d,rv_90d,rv_180d,rv_365d
0,SPY,2026-04-24,2026-04-24T20:00:00,2026-04-25T04:48:37.722000,7.29907,11.394565,10.689241,16.222181,15.743826,14.7243,13.294369,10.388955


### `/v1/cmiv`

Constant-maturity implied vol at fixed tenors (30d, 60d, 90d, ...). Use this when you want a clean IV time series that isn't polluted by rolling expiries.

In [ ]:
to_df(call("cmiv", ticker="SPY", start_date=MONTH_AGO, end_date=TODAY)).tail()

### `/v1/ivrank`

IV rank and IV percentile over 1-month, 3-month, and 1-year lookbacks. Fast screener for rich/cheap vol.

In [ ]:
to_df(call("ivrank", ticker=["SPY", "QQQ", "IWM", "TLT"]))

### `/v1/rviv`

Realized vol vs implied vol, with the variance risk premium.

VRP = 100 · ln(IV₃₀ / RV₃₀)

Positive VRP = IV above subsequent RV (the usual state for equity indices).

In [ ]:
to_df(call("rviv", ticker="SPY", start_date=MONTH_AGO, end_date=TODAY)).tail()

### `/v1/skew`

10-delta and 25-delta call/put skew plus historical percentiles. Useful for spotting stretched tails.

In [ ]:
to_df(call("skew", ticker="SPY", trade_date=MONTH_AGO)).head()

### `/v1/optionchain`

Full option chain — every strike, every expiry, IV + greeks.

**⚠️ EOD-only** and **heavy**. Always filter by `expiry_date` unless you really want the whole chain. Use `trade_date` for a past snapshot.

In [ ]:
chain = to_df(call("optionchain", ticker="SPY", trade_date=MONTH_AGO, expiry_date="2026-06-19"))
print(chain.shape)
chain.head()

### `/v1/cockpit`

Bundle: price, IV, returns, RVIV stats in a single response. Handy for dashboards — avoids N round-trips.

In [ ]:
cockpit = to_df(call("cockpit", ticker=["SPY", "QQQ"], start_date=MONTH_AGO, end_date=TODAY))
print(cockpit.shape)
cockpit.tail()

## Trade Ideas

### `/v1/trade-ideas`

Pre-built trade screens. **Gotcha:** `ideas`, `categories`, and `liquidity_levels` are **comma-separated strings** on this endpoint, not repeated params like `ticker`. Easy to get wrong.

In [ ]:
df = to_df(call("trade-ideas"))
print(df.shape)
df.head()

In [ ]:
to_df(call("trade-ideas",
             ticker=["SPY", "QQQ", "IWM", "TLT", "GLD", "USO"],
             liquidity_levels="High,Medium",
             exclude_earnings_weeks=2)).head()

## Bonus: CSV responses

Most endpoints accept `format=csv`. Use `raw=True` on `call()` to get the raw `Response`, then parse with pandas.

In [ ]:
from io import StringIO
resp = call("price", ticker="SPY", start_date=MONTH_AGO, end_date=TODAY, format="csv", raw=True)
pd.read_csv(StringIO(resp.text)).tail()

## Debugging

- **401 / 403** — check `.env` has `MOONTOWER_API_KEY` set and is loaded
- **400** — usually bad date format (`YYYY-MM-DD`) or conflicting date params
- **429** — rate limited; check `X-RateLimit-*` headers below
- **Empty `data`** on intraday `realvol`/`optionchain` — they're EOD-only
- **Unexpected shape on `trade-ideas`** — remember its list params are comma-separated strings

Rate-limit headers on any response:

In [ ]:
resp = call("price", ticker="SPY", raw=True)
{k: v for k, v in resp.headers.items() if "ratelimit" in k.lower()}